# 01 &mdash; A graph is three tensors

> **Reading order.** These notebooks are a course, not a reference. Each one
> assumes the previous. `01` builds the tensor vocabulary, `02` builds the layers,
> `03` trains them on Cora, `04` proves they are correct and breaks them
> deliberately, `05` moves to graph-level tasks.
>
> Everything imports the local `gnn_lab` package -- edit the package and re-run
> the cell. The self-contained Kaggle run lives in `kaggle/`.

**Goal of this notebook.** By the end you should be able to look at an
`edge_index` tensor and know exactly what graph it describes, and write a graph
convolution in five lines of plain PyTorch. No library, no abstraction.

That is the whole foundation. Every architecture in the rest of this repo is a
variation on it.

---

## 1. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))     # so `import gnn_lab` works from notebooks/

import torch
import torch.nn as nn
import torch.nn.functional as F

from gnn_lab import (GCNScratch, GraphSAGEScratch, GATScratch, GINScratch,
                     GCNConvScratch, SAGEConvScratch, GATConvScratch, GINConvScratch,
                     load_cora, load_mutag, triangle_square_dataset, karate_club,
                     toy_graph, path_graph, two_communities,
                     train_node, train_node_multi_seed, train_graph_classification,
                     evaluate_node, set_seed, get_device, count_parameters,
                     accuracy, macro_f1, majority_baseline,
                     mean_pairwise_cosine, dirichlet_energy)
from gnn_lab.ops import (scatter_add, scatter_mean, scatter_max, scatter_softmax,
                         gcn_norm, add_self_loops, remove_self_loops, to_undirected,
                         degree, dense_to_edge_index)
from gnn_lab.layers.pooling import collate_graphs, global_pool, graph_batches
from gnn_lab import equivalence, viz

DEVICE = get_device()
RESULTS = os.path.join("..", "results")
os.makedirs(RESULTS, exist_ok=True)

print("torch  :", torch.__version__)
print("device :", DEVICE)
print("PyG    :", "available" if equivalence.PYG_AVAILABLE else f"missing ({equivalence.PYG_IMPORT_ERROR})")
set_seed(0)


## 2. The tensor vocabulary

A graph is *not* an adjacency matrix in any implementation that scales. It is
three tensors:

| tensor | shape | meaning |
|---|---|---|
| `x` | `[N, F]` | one feature vector per node |
| `edge_index` | `[2, E]` | **row 0 = source** (the sender), **row 1 = destination** (the receiver) |
| `y` | `[N]` or `[G]` | labels, per node or per graph |

This is COO format: one column per *directed* edge. An undirected graph stores
both directions, so `E` is twice the number of edges you would draw on paper.

Why not a dense `[N, N]` matrix? Because real graphs are sparse. Cora has
2708 nodes and 5278 edges: a dense matrix would be 7.3M entries of which 0.2%
are non-zero, and every layer would cost `O(N^2 F)`. With `edge_index` the cost
is `O(E F)` -- and `E` grows roughly linearly in `N` for real networks, while
`N^2` obviously does not.

### The one convention that breaks everything

Messages are **gathered from `src`** and **scattered onto `dst`**. Get this
backwards and:

* the shapes are still valid,
* the model still trains,
* the loss still goes down,
* the accuracy still looks plausible,
* and the answer is quietly wrong.

There is no error to catch. This is why `gnn_lab` names the variables `src` and
`dst` everywhere instead of the terser `row` and `col`, and why
`tests/test_index_convention.py` enforces it statically.

In [ ]:
g = toy_graph()
x, edge_index = g["x"], g["edge_index"]

print("the graph:")
print("    0 -- 1 -- 2")
print("    |         |")
print("    3 ------- 4")
print()
print("x           :", tuple(x.shape))
print("edge_index  :", tuple(edge_index.shape), "  (2 rows x E columns)")
print()
print("column-by-column, each column is one directed edge:")
for e in range(edge_index.size(1)):
    s, d = int(edge_index[0, e]), int(edge_index[1, e])
    print(f"  edge {e:2d}:  src {s} -> dst {d}")

print()
deg = degree(edge_index[1], num_nodes=5)
print("in-degree of each node:", deg.tolist())
print("-> every node has degree 2. That makes this graph 'regular', which is")
print("   why the expected values computed later in this notebook are exact.")

### Undirected graphs store both directions

Notice every edge above appears twice: `(0,1)` and `(1,0)`. That is not
redundancy, it is the price of the format -- a single directed list cannot
express "these two nodes are connected" without both entries.

`gnn_lab.ops.to_undirected` symmetrises *and* de-duplicates, which matters more
than you would think (see notebook 03 for a real dataset where it changes the
answer).

In [ ]:
ei = torch.tensor([[0, 1, 1, 0, 2],
                   [1, 0, 2, 0, 1]])          # note (0,1) and (1,0) both appear, plus a dup
print("input edge_index :")
print(ei.tolist())
print("columns:", ei.size(1))

u = to_undirected(ei, num_nodes=3)
print("\nafter to_undirected:")
print(u.tolist())
print("columns:", u.size(1), "-> 3 unique undirected edges stored in both directions")

loops = add_self_loops(u, num_nodes=3)
print("\nafter add_self_loops:", loops.size(1), "columns")
print("self-loops appended :", remove_self_loops(loops).size(1) == u.size(1))

### From a dense matrix, and back

`dense_to_edge_index` is the bridge. Useful for tiny experiments and for
converting other people's code; never use it for anything large.

In [ ]:
adj = torch.tensor([[0, 1, 0, 1],
                    [1, 0, 1, 0],
                    [0, 1, 0, 1],
                    [1, 0, 1, 0]])
print("dense adjacency (a 4-cycle):")
print(adj.tolist())

ei = dense_to_edge_index(adj)
print("\nedge_index:")
print(ei.tolist())

rebuilt = torch.zeros(4, 4, dtype=torch.long)
rebuilt[ei[0], ei[1]] = 1
print("\nround-trip identical:", bool(torch.equal(rebuilt, adj)))

## 3. Visualising a real small graph

Zachary's karate club: 34 members of a university karate club in 1977, 78
friendship ties, and a documented split into two factions after a dispute over
pricing. It is the standard "can the model see social structure?" dataset.

Note the homophily measurement below -- 67 of 78 ties are *within* a faction.
That is why a GNN does well here, and it is the assumption that breaks on
heterophilous graphs.

In [ ]:
k = karate_club()
viz.draw_graph(k["edge_index"], labels=k["y"],
               title="Zachary's karate club (34 nodes, 78 ties, coloured by faction)",
               path=f"{RESULTS}/01_karate.png", node_size=200)

same = (k["y"][k["edge_index"][0]] == k["y"][k["edge_index"][1]]).float().mean()
print(f"\nnodes: {k['x'].shape[0]}   directed edges: {k['edge_index'].size(1)}")
print(f"homophily (fraction of edges joining the same faction): {same:.3f}")
print("-> a neighbour is very likely to share your label, which is exactly the")
print("   assumption message passing exploits.")

## 4. Receptive field: why GNNs are shallow

After `k` layers of message passing, a node has aggregated information from
everything within `k` hops. That is a feature and a limitation at the same time.

The path graph makes it unambiguous: node `i` can only ever see nodes
`i-k ... i+k` after `k` layers. Set `k` large enough and *every* node sees the
whole graph, so every node ends up with the same representation. That is
over-smoothing, and notebook 04 measures it properly.

Here you can watch the receptive field grow.

In [ ]:
p = path_graph(n=11, feature_dim=1)
ei = p["edge_index"]

# give node 5 a unique signal and see how far it travels after k layers
h = torch.zeros(11, 1)
h[5] = 1.0

print("a unit signal placed on node 5, diffused with plain mean aggregation:")
print(f"{'layers':>7} | " + " ".join(f"{i:>5}" for i in range(11)))
print("-" * 78)
for k in range(0, 7):
    print(f"{k:>7} | " + " ".join(f"{float(v):5.2f}" for v in h.squeeze()))
    # one step of mean aggregation over neighbours
    h = scatter_mean(h.index_select(0, ei[0]), ei[1], dim_size=11)

print()
print("After k layers the signal has reached exactly nodes 5-k .. 5+k.")
print("By layer 6 it has touched almost the whole graph, and every node is")
print("converging to the same value -- that is over-smoothing in miniature.")

## 5. The five-line graph convolution

Now the payoff. Here is a complete GCN layer, with a linear map set to the
identity so the arithmetic is readable:

1. `gcn_norm` -- add self-loops and compute the `1/sqrt(d_i d_j)` coefficient per edge
2. `x @ W` -- an ordinary linear map, no graph involved
3. `h.index_select(0, src) * norm` -- **MESSAGE**: gather from the sender
4. `scatter_add(msg, dst, N)` -- **AGGREGATE**: unsorted group-by onto the receiver
5. `+ bias` -- **UPDATE**

Because the toy graph is 2-regular, every node has degree 3 after self-loops,
so every coefficient is exactly `1/3`. That means the expected output can be
computed on paper.

In [ ]:
g = toy_graph()
x, ei = g["x"].double(), g["edge_index"]
N = x.size(0)

# --- 1. normalisation coefficients ---------------------------------------
ei_loops, norm = gcn_norm(ei, N)
print(f"edges: {ei.size(1)} original + {N} self-loops = {ei_loops.size(1)}")
print(f"norm coefficients, unique values: {sorted({round(float(v), 6) for v in norm})}")
print("  -> all exactly 1/3, because every node has degree 3 with its loop")

# --- 2. linear map --------------------------------------------------------
W = torch.eye(2, dtype=torch.float64)
h = x @ W

# --- 3. MESSAGE: gather from the SOURCE, scale ----------------------------
src, dst = ei_loops[0], ei_loops[1]
msg = h.index_select(0, src) * norm.unsqueeze(-1)
print(f"\nmessage tensor: {tuple(msg.shape)} -- one row per directed edge")

# --- 4. AGGREGATE: scatter onto the DESTINATION ---------------------------
out = scatter_add(msg, dst, N)

# --- 5. UPDATE ------------------------------------------------------------
bias = torch.zeros(2, dtype=torch.float64)
out = out + bias

print("\nresult (W = identity, bias = 0):")
for i, row in enumerate(out.tolist()):
    print(f"  node {i}: [{row[0]:.4f}, {row[1]:.4f}]")

### Checking it against arithmetic done by hand

For node 0, the neighbours are 1 and 3, plus the self-loop:

```
out[0] = (x0 + x1 + x3) / 3
       = ([1,0] + [0,1] + [2,0]) / 3
       = [3, 1] / 3
       = [1.0000, 0.3333]
```

Let the machine confirm it, for all five nodes:

In [ ]:
expected = torch.tensor([[3, 1],
                         [2, 2],
                         [1, 4],
                         [3, 2],
                         [3, 3]], dtype=torch.float64) / 3.0

print("hand-computed expectation:")
for i, row in enumerate(expected.tolist()):
    nbrs = {0: "x0+x1+x3", 1: "x1+x0+x2", 2: "x2+x1+x4", 3: "x3+x0+x4", 4: "x4+x2+x3"}[i]
    print(f"  node {i} = ({nbrs})/3 = [{row[0]:.4f}, {row[1]:.4f}]")

print("\nmatches the five lines above:", bool(torch.allclose(out, expected, atol=1e-12)))

# and the library layer must agree too
conv = GCNConvScratch(2, 2).double().eval()
with torch.no_grad():
    conv.lin.weight.copy_(torch.eye(2, dtype=torch.float64))
    conv.bias.zero_()
    out_layer = conv(x, ei, N)
print("GCNConvScratch agrees with the hand-written version:",
      bool(torch.allclose(out_layer, expected, atol=1e-12)))

## 6. The scatter primitives, checked against brute force

Everything above rests on `scatter_add`, `scatter_mean`, `scatter_max` and
`scatter_softmax`. Each is an *unsorted group-by*, implemented with
`index_add_` / `index_reduce_` -- a single fused CUDA kernel, not a Python loop.

Let's verify them against exactly that Python loop, which is the only
definition-level check that means anything.

In [ ]:
for r in equivalence.check_scatter_vs_loop():
    print(r.line())

### What `scatter_softmax` is for

Softmax *within each group* -- not globally. This is the operation that turns
GAT's raw scores into a proper attention distribution, where each node's
incoming weights sum to 1.

Two details worth internalising:

* The per-group max is subtracted before exponentiating, exactly like the
  textbook softmax trick.
* That max is `.detach()`-ed. This is **mathematically exact**, not an
  approximation: softmax is invariant to a constant shift of its inputs, so the
  shift contributes precisely zero to the gradient. `tests/test_ops.py` verifies
  the gradient matches a reference that keeps the max attached.

In [ ]:
index = torch.tensor([0, 0, 0, 1, 1, 2])
scores = torch.tensor([[2.0], [1.0], [0.5], [3.0], [-1.0], [5.0]])

att = scatter_softmax(scores, index, dim_size=3)
print("raw scores grouped by target node:")
for k in range(3):
    m = index == k
    print(f"  node {k}: raw {scores[m].squeeze().tolist()}  ->  softmax {att[m].squeeze().tolist()}")

totals = scatter_add(att, index, 3)
print("\nsum within each group:", totals.squeeze().tolist())
print("-> each node's incoming attention sums to 1. A global softmax could not do this.")

## 7. Summary

| concept | where it lives |
|---|---|
| graph = `x`, `edge_index`, `y` | `gnn_lab.data` |
| `src` gathers, `dst` scatters | `gnn_lab.ops`, enforced by `tests/test_index_convention.py` |
| group-by = `index_add_` | `scatter_add` |
| mean / max / softmax reductions | `scatter_mean`, `scatter_max`, `scatter_softmax` |
| GCN coefficients `1/sqrt(d_i d_j)` | `gcn_norm` |
| receptive field grows with depth | section 4, measured properly in notebook 04 |

**Next:** `02_message_passing_layers.ipynb` -- turning those five lines into
GCN, GraphSAGE, GAT and GIN, and seeing that they are the same function with
three knobs turned differently.